# Hierarchical Routing Pipeline

This notebook isolates the Domain Classifier (Level 1 Router) of the hierarchical architecture. 
It trains a fast, lightweight classifier to determine the task domain of a user query, which can then be used to route the query to a specialized LoRA expert or to filter tools before passing the context to the main LLM.

In [1]:
# === IMPORT LIBRARIES ===
import json
import pickle
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [ ]:
# === CONFIGURATION ===
# Note: Hierarchical Routing uses a fast ML classifier (TF-IDF + Logistic Regression)
# instead of an embedding model, so we tune these hyperparameters instead of top_k.

MAX_FEATURES = 10000        # Max vocabulary size for TF-IDF
MAX_ITER = 1000             # Max iterations for Logistic Regression
CLASS_WEIGHT = "balanced"   # Weighting strategy for class imbalance

DATA_PATH = "../data/train.jsonl"
MODEL_OUTPUT_PATH = "../output/models/domain_router.pkl"


In [2]:
# 1. Load Data to train the router
# In a real scenario, you'd extract the ground-truth 'domain' based on the 'answer' tool name
queries = []
domains = []

print("Loading training data for the router...")
with open(DATA_PATH, "r") as f:
    for line in f:
        sample = json.loads(line)
        correct_tool = sample["options"][sample["answer"]]["name"]
        
        # Simple heuristic to extract domain from tool name (e.g. 'flight_search' -> 'flight')
        domain = correct_tool.split("_")[0] 
        
        queries.append(sample["full_context"])
        domains.append(domain)

Loading training data for the router...


In [3]:
# 2. Build and train a lightweight TF-IDF + Logistic Regression Classifier
print("Training the Domain Router...")
router = Pipeline([
    (tfidf, TfidfVectorizer(max_features=MAX_FEATURES)),
    ('clf', LogisticRegression(max_iter=MAX_ITER, class_weight=CLASS_WEIGHT))
])

router.fit(queries, domains)
print(f"Trained router on {len(queries)} samples across {len(set(domains))} domains.")

Training the Domain Router...
Trained router on 13587 samples across 405 domains.


In [ ]:
# 3. Save the router for the Inference Pipeline
import os
os.makedirs("../output/models", exist_ok=True)
with open(MODEL_OUTPUT_PATH, "wb") as f:
    pickle.dump(router, f)
    
print("Router saved to output/models/domain_router.pkl")

### How to use this in `98_generate_submission.ipynb`:
```python
with open('../output/models/domain_router.pkl', 'rb') as f:
    domain_router = pickle.load(f)

def route_query(context):
    # 1. Predict Domain
    predicted_domain = domain_router.predict([context])[0]
    
    # 2. Route to specialized model (or filter tools)
    if predicted_domain == 'flight':
        return flight_expert_model.generate(context)
    else:
        return general_model.generate(context)
```